In [ ]:
import torch
from transformers import AutoModelForTokenClassification
from transformers import AutoTokenizer




MODEL_NAME = "HUMADEX/german_medical_ner"


In [35]:
def get_device():
    """
    Determine the best available computation device.

    Returns
    -------
    torch.device
        CUDA, Apple MPS, or CPU device.
    """
    if torch.cuda.is_available():
        return torch.device("cuda")

    if torch.backends.mps.is_available():
        return torch.device("mps")

    return torch.device("cpu")

In [ ]:

def load_medical_ner(
    model_name: str = MODEL_NAME,
):
    """
    Load the tokenizer and medical NER model.

    Parameters
    ----------
    model_name : str
        Hugging Face model identifier.

    Returns
    -------
    tuple
        Tokenizer, model and computation device.
    """
    device = get_device()

    tokenizer = AutoTokenizer.from_pretrained(
        model_name
    )

    model = AutoModelForTokenClassification.from_pretrained(
        model_name
    )

    model.to(device)
    model.eval()

    return tokenizer, model, device


def predict_tokens(
    query: str,
    tokenizer,
    model,
    device,
):
    """
    Predict NER labels and character spans for tokens.

    Parameters
    ----------
    query : str
        Input query.
    tokenizer
        Hugging Face tokenizer.
    model
        Token classification model.
    device : torch.device
        Computation device.

    Returns
    -------
    list of dict
        Token predictions with character offsets.
    """
    encoded = tokenizer(
        query,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        max_length=512,
    )

    offsets = encoded.pop("offset_mapping")[0]

    model_inputs = {
        key: value.to(device)
        for key, value in encoded.items()
    }

    with torch.no_grad():
        outputs = model(**model_inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1,
    )

    predictions = torch.argmax(
        outputs.logits,
        dim=-1,
    )[0]

    tokens = tokenizer.convert_ids_to_tokens(
        model_inputs["input_ids"][0]
    )

    results = []

    for token, prediction, probability, offset in zip(
        tokens,
        predictions,
        probabilities[0],
        offsets,
    ):
        start, end = offset.tolist()

        if start == end:
            continue

        label = model.config.id2label[
            prediction.item()
        ]

        results.append(
            {
                "token": token,
                "label": label,
                "score": float(
                    probability[prediction]
                ),
                "start": start,
                "end": end,
            }
        )

    return results


def group_subtokens(
    query: str,
    predictions,
):
    """
    Group WordPiece subtokens into complete words.

    Parameters
    ----------
    query : str
        Original query.
    predictions : list of dict
        Token predictions.

    Returns
    -------
    list of dict
        Predictions grouped into words.
    """
    words = []
    current = None

    for prediction in predictions:
        start = prediction["start"]
        end = prediction["end"]

        text = query[start:end]

        if current is None:
            current = {
                "text": text,
                "start": start,
                "end": end,
                "predictions": [prediction],
            }
            continue

        previous_end = current["end"]
        gap = query[previous_end:start]

        is_subtoken = prediction["token"].startswith("##")

        if is_subtoken or gap == "":
            current["text"] += text
            current["end"] = end
            current["predictions"].append(prediction)
        else:
            words.append(current)

            current = {
                "text": text,
                "start": start,
                "end": end,
                "predictions": [prediction],
            }

    if current is not None:
        words.append(current)

    return words


def get_word_label(
    word,
):
    """
    Determine the dominant NER label for a grouped word.

    Parameters
    ----------
    word : dict
        Grouped word containing token predictions.

    Returns
    -------
    tuple
        Entity label and confidence score.
    """
    scores = {}

    for prediction in word["predictions"]:
        label = prediction["label"]

        if label == "O":
            continue

        entity_type = label.split("-", 1)[-1]

        scores.setdefault(
            entity_type,
            [],
        )

        scores[entity_type].append(
            prediction["score"]
        )

    if not scores:
        return "O", 0.0

    label = max(
        scores,
        key=lambda key: sum(scores[key]),
    )

    score = sum(scores[label]) / len(
        scores[label]
    )

    return label, score


def extract_medical_entities(
    query: str,
    tokenizer,
    model,
    device,
):
    """
    Extract medical entities as complete words or text spans.

    Parameters
    ----------
    query : str
        German medical query.
    tokenizer
        Hugging Face tokenizer.
    model
        Token classification model.
    device : torch.device
        Computation device.

    Returns
    -------
    list of dict
        Extracted medical entities.
    """
    predictions = predict_tokens(
        query=query,
        tokenizer=tokenizer,
        model=model,
        device=device,
    )

    words = group_subtokens(
        query=query,
        predictions=predictions,
    )

    entities = []

    for word in words:
        label, score = get_word_label(word)

        if label == "O":
            continue

        entities.append(
            {
                "text": word["text"],
                "label": label,
                "score": score,
                "start": word["start"],
                "end": word["end"],
            }
        )

    return entities


def extract_medical_keywords(
    query: str,
    tokenizer,
    model,
    device,
):
    """
    Extract medical keywords grouped by entity type.

    Parameters
    ----------
    query : str
        German medical query.
    tokenizer
        Hugging Face tokenizer.
    model
        Token classification model.
    device : torch.device
        Computation device.

    Returns
    -------
    dict
        Medical keywords grouped by entity type.
    """
    entities = extract_medical_entities(
        query=query,
        tokenizer=tokenizer,
        model=model,
        device=device,
    )

    keywords = {
        "PROBLEM": [],
        "TEST": [],
        "TREATMENT": [],
    }

    for entity in entities:
        label = entity["label"]

        if label in keywords:
            keywords[label].append(
                entity["text"]
            )

    return keywords




Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Using device: cpu

Ich habe starke Kopfschmerzen und Schwindel und brauche ein MRT.
{'PROBLEM': ['starke', 'Kopfschmerzen', 'Schwindel'], 'TEST': ['ein', 'MRT.'], 'TREATMENT': []}

Ich habe starkeKopfschmerzen und Schwindel.
{'PROBLEM': ['starkeKopfschmerzen', 'Schwindel.'], 'TEST': [], 'TREATMENT': []}

Ich habe starke KopfschmerzenundSchwindel.
{'PROBLEM': ['starke', 'KopfschmerzenundSchwindel.'], 'TEST': [], 'TREATMENT': []}


In [37]:
queries = [
        "Ich habe seit mehreren Tagen starke Kopfschmerzen "
        "und Schwindel und brauche ein MRT.",
        "Knie-OP Chirurgie in München",
        "Ich habe starke Kopfschmerzen und Schwindel und brauche ein MRT.",
        "Ich habe starkeKopfschmerzen und Schwindel.",
        "Ich habe starke KopfschmerzenundSchwindel.",
]

In [38]:
tokenizer, model, device = load_medical_ner()

print(f"Using device: {device}")


for query in queries:
        print()
        print(query)

        keywords = extract_medical_keywords(
            query=query,
            tokenizer=tokenizer,
            model=model,
            device=device,
        )

        print(keywords)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Using device: cpu

Ich habe seit mehreren Tagen starke Kopfschmerzen und Schwindel und brauche ein MRT.
{'PROBLEM': ['starke', 'Kopfschmerzen', 'Schwindel'], 'TEST': ['ein', 'MRT.'], 'TREATMENT': []}

Knie-OP Chirurgie in München
{'PROBLEM': ['in', 'München'], 'TEST': [], 'TREATMENT': ['Knie-OP', 'Chirurgie']}

Ich habe starke Kopfschmerzen und Schwindel und brauche ein MRT.
{'PROBLEM': ['starke', 'Kopfschmerzen', 'Schwindel'], 'TEST': ['ein', 'MRT.'], 'TREATMENT': []}

Ich habe starkeKopfschmerzen und Schwindel.
{'PROBLEM': ['starkeKopfschmerzen', 'Schwindel.'], 'TEST': [], 'TREATMENT': []}

Ich habe starke KopfschmerzenundSchwindel.
{'PROBLEM': ['starke', 'KopfschmerzenundSchwindel.'], 'TEST': [], 'TREATMENT': []}
